# Climbing Stairs II - LeetCode 3693

You start at position `0` and want to reach position `n`. You may jump `1`, `2`, or `3` stairs.

- From position `i`, you can jump to `i + 1`, `i + 2`, or `i + 3`
- The cost of landing at position `j` is `costs[j - 1]`
- The jump cost is `(j - i) * (j - i)`

**Recurrence:**
```text
solve(i) = min for 1 <= jump <= 3:
           jump * jump + costs[i + jump - 1] + solve(i + jump)
```

`solve(n) = 0` because no more stairs need to be climbed. The answer is `solve(0)`.

## Approaches covered

1. Plain recursion
2. Recursion with memoization
3. Memoization using a jump loop
4. Bottom-up tabulation
5. Bottom-up dynamic programming with constant space

## Approach 1: Plain Recursion

The function checks every valid jump of 1, 2, or 3 steps and returns the minimum total cost.

**State definition:**
- `solve(i)` = minimum cost required to move from position `i` to position `n`

**Base case:**
- `solve(n) = 0`

**Transition:**
- For each `jump` from `1` to `3`, calculate:
`jump * jump + costs[i + jump - 1] + solve(i + jump)`
- Return the minimum valid result

- Time: `O(3^n)` in the worst case
- Auxiliary space: `O(n)` due to recursion-stack depth

In [ ]:
//recursion
class Solution {
public:
    int solve(int i, int n, vector<int>& costs) {
        if (i == n)
            return 0;

        int one = INT_MAX;
        int two = INT_MAX;
        int three = INT_MAX;

        if (i + 1 <= n) {
            one = 1 + costs[i] + solve(i + 1, n, costs);//1-indexed so cost[j] = costs[i]
        }

        if (i + 2 <= n) {
            two = 4 + costs[i+1] + solve(i + 2, n, costs);
        }
        if (i + 3 <= n) {
            three = 9 + costs[i+2] + solve(i + 3, n, costs);
        }
        return min({one, two, three});
    }
    int climbStairs(int n, vector<int>& costs) { return solve(0, n, costs); }
};

## Approach 2: Recursion + Memoization (Top-Down DP)

This is the recursive solution with a memo table. Once `solve(i)` has been calculated, its result is reused for later calls.

**State definition:**
- `memo[i]` stores the minimum cost to move from position `i` to position `n`
- `memo[i] == -1` means the state has not been calculated

**Base cases:**
- `i == n -> 0`, so no jump or landing cost is required

**Transition:**
- Calculate costs for jumps of `1`, `2`, and `3` steps
- Store the minimum answer in `memo[i]`
- Direction: **Top-down** - move from larger position `i` toward position `n`

- Time: `O(n)` because each position is solved once
- Auxiliary space: `O(n)` for the memo table and recursion stack

In [ ]:
//memoization --> TC: O(n) || SC:O(n)
class Solution {
public:
    int solve(int i, int n, vector<int>& costs, vector<int>& memo) {
        if (n == i)
            return 0;
        if (memo[i] != -1)
            return memo[i];

        int cost = INT_MAX;

        int j = i + 1;
        if (j <= n) {
            cost = costs[j - 1] + (j - i) * (j - i) + solve(j, n, costs, memo);
        }
        j = i + 2;
        if (j <= n) {
            cost = min(cost, costs[j - 1] + (j - i) * (j - i) +
                                 solve(j, n, costs, memo));
        }
        j = i + 3;
        if (j <= n) {
            cost = min(cost, costs[j - 1] + (j - i) * (j - i) +
                                 solve(j, n, costs, memo));
        }
        return memo[i] = cost;
    }
    int climbStairs(int n, vector<int>& costs) {
        vector<int> memo(n + 1, -1);
        return solve(0, n, costs, memo);
    }
};

## Approach 3: Memoization Using a Jump Loop

This implementation uses the same top-down memoization idea, but replaces the three separate jump branches with a loop.

**State definition:**
- `memo[i]` stores the minimum cost to reach position `n` from position `i`

**Transition:**
- Loop through `j = i + 1` to `min(i + 3, n)`
- Calculate `jump = (j - i) * (j - i)`
- Candidate cost: `costs[j - 1] + jump + solve(j, n, costs, memo)`
- Store the minimum candidate in `memo[i]`

- Time: `O(n)`
- Auxiliary space: `O(n)`

In [ ]:
//memoization using loop
class Solution {
public:
    int solve(int i, int n, vector<int>& costs, vector<int>& memo) {
        if (n == i)
            return 0;
        if (memo[i] != -1)
            return memo[i];

        int ans = INT_MAX;
        for (int j = i + 1; j <= min(i + 3, n); j++) {
            int jump = (j - i) * (j - i);
            int cost = costs[j - 1] + jump + solve(j, n, costs, memo);
            ans = min(ans, cost);
        }
        return memo[i] = ans; // store min cost
    }
    int climbStairs(int n, vector<int>& costs) {
        vector<int> memo(n + 1, -1);
        return solve(0, n, costs, memo);
    }
};

## Approach 4: Tabulation (Bottom-Up DP)

Instead of recursion, we calculate the minimum cost from each position starting at `n - 1` and move backward to `0`.

**State definition:**
- `tab[i]` stores the minimum cost to move from position `i` to position `n`

**Initialization and transition:**
- `tab[n] = 0`
- For `i = n - 1` down to `0`, consider every valid `j` from `i + 1` to `min(i + 3, n)`
- Candidate: `jump + costs[j - 1] + tab[j]`
- Store the minimum candidate in `tab[i]`
- Final answer: `tab[0]`
- Direction: **Bottom-up** - states closer to `n` are calculated first

- Time: `O(n)`
- Auxiliary space: `O(n)`

In [ ]:
//using tabulation--> TC:O(n) || SC:O(n)
class Solution {
public:
    int climbStairs(int n, vector<int>& costs) {
        vector<int>tab(n+1 , -1);
        tab[n] = 0;
        
        for(int i = n-1 ; i>=0 ; i--){

            int ans = INT_MAX;

            for(int j = i+1; j <= min(i+3,n) ; j++){
                int jump = (j-i)*(j-i);
                int cost = jump + costs[j-1] + tab[j];
                ans = min(cost,ans);
            }
            tab[i] = ans;
        }
       return tab[0]; 
    }
};

## Approach 5: Bottom-Up DP with Constant Space

Because each state depends only on the next 1, 2, or 3 positions, we can replace the DP array with three variables.

**Variables used:**
- `a` stores `tab[i + 1]`
- `b` stores `tab[i + 2]`
- `c` stores `tab[i + 3]`
- `ans` stores the minimum result for the current position

**Update pattern:**
- Calculate `ans` using `a`, `b`, or `c` according to the jump length
- Then shift the values backward:
```text
c = b
b = a
a = ans
```

- Base case: position `n` has cost `0`, represented by `a = 0` at the end of the first iteration setup
- Final answer: `a` after the loop reaches position `0`
- Time: `O(n)`
- Auxiliary space: `O(1)`

In [ ]:
// using space optimization --> TC:O(n) || SC:O(1)
class Solution {
public:
    int climbStairs(int n, vector<int>& costs) {

        int a = 0;
        int b = INT_MAX;
        int c = INT_MAX;
        for (int i = n - 1; i >= 0; i--) {
            int ans = INT_MAX;

            for (int j = i + 1; j <= min(i + 3, n); j++) {
                int jump = (j - i) * (j - i);
                int nextCost;//storing dp[j] as nextCost
                if (j == i + 1)
                    nextCost = a;
                else if (j == i + 2)
                    nextCost = b;
                else
                    nextCost = c;
                int cost = jump + costs[j - 1] + nextCost;
                ans = min(cost, ans);
            }
            c = b;
            b = a;
            a = ans;
        }
        return a;
    }
};